# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.3 - Fine-Tuning

In this notebook, we are going to fine-tune Qwen 2.5 - 7B Instruct model, using the dataset and reward model set up in the previous notebooks

---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"
base_model_jumpstart_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_shortname = "qwen25-7b"

s3_output_path = f"s3://{bucket_name}/{project_prefix}-{base_model_shortname}/"

---

## Set up the model package group

A ModelPackageGroup is a group of versioned models in the SageMaker Model Registry. You can view your ModelPackageGroups in SageMaker AI Studio when selecting Models in the left navigation bar and choosing the "My models" tab.

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name=f"{project_prefix}-{base_model_shortname}"

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless RLAIF customization",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

---

## Set up the Serverless RLAIF Training Job
We configure the RLAIF training job by bringing together all the components defined earlier: the base model to fine-tune, the registered training and validation datasets, the reward model that will score candidate responses, and the reward prompt that defines our evaluation criteria. The `RLAIFTrainer` orchestrates the reinforcement learning loop where the base model generates responses, the reward model scores them using our human-likeness prompt, and the policy is updated accordingly.

### Define the `RLAIFTrainer` instance

In [ ]:
from sagemaker.train.rlaif_trainer import RLAIFTrainer
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.evaluator import Evaluator

training_dataset = DataSet.get(name=f"{project_prefix}-train")
val_dataset = DataSet.get(name=f"{project_prefix}-eval")

base_model_id = base_model_jumpstart_id
reward_model_id = "openai.gpt-oss-120b-1:0"
reward_prompt = Evaluator.get(name=f"{project_prefix}-reward-prompt")

trainer = RLAIFTrainer(
    model=base_model_id,
    model_package_group=model_package_group_name,
    reward_model_id=reward_model_id,
    reward_prompt=reward_prompt.arn,
    training_dataset=training_dataset,
    s3_output_path=s3_output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True
)

### Inspect and configure the training hyperparameters
We will set the number of epochs to 1, so the training finishes in a reasonable timeframe for this workshop. All the other hyperparameters can be left at their defaults.

In [ ]:
from rich.pretty import pprint

trainer.hyperparameters.max_epochs = 1
pprint(trainer.hyperparameters.to_dict())

### Start the training job
Launch the RLAIF training job. Setting `wait=False` runs it asynchronously so we can monitor progress separately without blocking the notebook.

In [ ]:
training_job = trainer.train(wait=False)
print(f"Created training job: {training_job.training_job_name}")